In [33]:
!pip install ipywidgets
!pip install folium
!pip install geopandas
!pip install shapely
!pip install pandas
!pip install requests
!pip install ipyleaflet


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [8]:
from skyfield.api import load, EarthSatellite, Topos
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import requests

def fetch_tle_text(url):
    """Fetches TLE data from a given URL and returns the raw text."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching TLE data from {url}: {e}")
        return ""

# Define TLE sources
tle_sources = {
    'LANDSAT 8': 'https://celestrak.org/NORAD/elements/resource.txt',
    'LANDSAT 9': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2A': 'https://celestrak.org/NORAD/elements/resource.txt',  
    'SENTINEL-2B': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2C': None, # Manually provided TLE as it was not available on the link
}

# Manually provided TLE for SENTINEL-2C
sentinel_2c_tle = [
    "1 60989U 24157A   25090.79518797  .00000292  00000-0  12798-3 0  9993",
    "2 60989  98.5659 167.0180 0001050  95.0731 265.0572 14.30814009 29727"
]

# Load time scale
ts = load.timescale()

# Load satellites into a dictionary
satellites = {}

for name, url in tle_sources.items():
    if name == 'SENTINEL-2C':
        line1, line2 = sentinel_2c_tle
        satellites[name] = EarthSatellite(line1, line2, name, ts)
        print(f"Loaded TLE data for {name}")
    else:
        tle_text = fetch_tle_text(url)
        tle_lines = tle_text.splitlines()
        for i in range(len(tle_lines) - 2):
            if name in tle_lines[i]:  # Look for the satellite name in the TLE file
                line1, line2 = tle_lines[i+1], tle_lines[i+2]
                satellites[name] = EarthSatellite(line1, line2, name, ts)
                print(f"Loaded TLE data for {name}")
                break

# Verify loaded satellites
if not satellites:
    print("No satellites were loaded. Check TLE sources.")

# Function to calculate precise overpass times using TLE
def find_overpasses(lat, lon, start_date, end_date, satellite):
    """Finds satellite overpass times when the nadir is closest to (lat, lon)."""
    if satellite not in satellites:
        print(f"Error: {satellite} TLE not available")
        return []

    sat = satellites[satellite]
    observer = Topos(latitude_degrees=lat, longitude_degrees=lon)
    
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')

    results = []
    dt = start_dt

    while dt <= end_dt:
        # Generate time steps every 1 seconds for better accuracy. For faster processing change time from one second to more.
        t = ts.utc(dt.year, dt.month, dt.day, 0, 0, np.arange(0, 86400, 1))
        
        # Compute nadir position (ground track)
        subpoint = sat.at(t).subpoint()
        latitudes = subpoint.latitude.degrees
        longitudes = subpoint.longitude.degrees

        # Find the closest approach to the given location
        distances = np.sqrt((latitudes - lat) ** 2 + (longitudes - lon) ** 2)
        min_index = np.argmin(distances)
        closest_time = t[min_index].utc_datetime()

        # Compute satellite position relative to observer
        topocentric = (sat - observer).at(t[min_index])
        alt, az, distance = topocentric.altaz()

        # Prepare the output data
        if distances[min_index] < 0.5:  # Threshold: 0.5° (1° = 111km so 0.5° = 55.5 km accuracy)
            results.append({
                'date': closest_time.strftime('%Y-%m-%d %H:%M:%S'),
                'Satellite': satellite,
                'Lat (DEG)': latitudes[min_index],
                'Lon (DEG)': longitudes[min_index],
                'Sat. Azi. (deg)': az.degrees,
                'Sat. Elev. (deg)': alt.degrees,
                'Range (km)': distance.km,
            })

        # Move to the next possible overpass (repeat cycle)
        dt += timedelta(days=1)

    return results

# Function to get satellite overpass times with additional parameters
def get_precise_overpass(lat, lon, start_date, end_date):
    overpasses = []
    for sat_name in satellites.keys():
        overpasses.extend(find_overpasses(lat, lon, start_date, end_date, sat_name))
    
    return pd.DataFrame(overpasses)

# Salzburg hbf location
lat = 47.81306
lon = 13.04667
start_date = '2025-02-01'
end_date = '2025-06-08'

overpass_times = get_precise_overpass(lat, lon, start_date, end_date)
print(overpass_times)

today = datetime.today().date();
print(today)
daterange = f"prediction_{start_date}_to_{end_date}" 
overpass_times.to_csv(f"Outputs/prediction_{today}.csv", index=False)
print("Data added to CSV")


Loaded TLE data for LANDSAT 8
Loaded TLE data for LANDSAT 9
Loaded TLE data for SENTINEL-2A
Loaded TLE data for SENTINEL-2B
Loaded TLE data for SENTINEL-2C
                   date    Satellite  Lat (DEG)  Lon (DEG)  Sat. Azi. (deg)  \
0   2025-02-04 20:44:12    LANDSAT 8  47.786788  13.018937       215.424368   
1   2025-02-20 20:45:56    LANDSAT 8  47.625866  12.592435       238.748035   
2   2025-02-25 09:55:39    LANDSAT 8  47.725953  13.236868       124.127377   
3   2025-03-13 09:56:44    LANDSAT 8  47.839905  12.959021       294.495566   
4   2025-03-17 20:41:36    LANDSAT 8  47.975249  13.495405        61.571730   
..                  ...          ...        ...        ...              ...   
93  2025-05-12 10:17:28  SENTINEL-2C  47.910354  12.774794       298.111699   
94  2025-05-20 20:58:40  SENTINEL-2C  47.965925  13.356049        53.574388   
95  2025-05-22 10:17:12  SENTINEL-2C  47.881972  12.827651       295.140912   
96  2025-05-30 20:58:20  SENTINEL-2C  47.964157  13.43